In [1]:
import pandas as pd

In [2]:
# Load dataset
df = pd.read_csv("../preparation/newcolumns.csv")
print(df.shape)
print(df.columns)

(16578, 32)
Index(['GAME_DATE', 'GAME_ID', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'TEAM_NAME', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE',
       'OPP', 'WIN', 'HOME'],
      dtype='str')


In [3]:
# Ensure data is sorted chronologically
df = df.sort_values(["TEAM_ID", "GAME_DATE"]).reset_index(drop=True)

In [4]:
df['PREV_WIN'] = df.groupby('TEAM_ID')['WIN'].shift(1)
df['PREV_PTS'] = df.groupby('TEAM_ID')['PTS'].shift(1)
df['PREV_PLUSMINUS'] = df.groupby('TEAM_ID')['PLUS_MINUS'].shift(1)

In [5]:
# # 3. Create a lookup dataframe containing these previous stats for every team per game
# opp_prev_df = df[[
#     "GAME_ID",
#     "TEAM_ABBREVIATION",
#     "PREV_WIN",
#     "PREV_PTS",
#     "PREV_PLUSMINUS",
# ]].copy()

In [6]:
# # Rename columns to represent the opposition
# opp_prev_df = opp_prev_df.rename(
#     columns={
#         "TEAM_ABBREVIATION": "OPP",
#         "PREV_WIN": "OPP_PREV_WIN",
#         "PREV_PTS": "OPP_PREV_PTS",
#         "PREV_PLUSMINUS": "OPP_PREV_PLUSMINUS",
#     }
# )

In [7]:
print(df["TEAM_ABBREVIATION"].dtype)
print(df["TEAM_ABBREVIATION"].head(10))

str
0    ATL
1    ATL
2    ATL
3    ATL
4    ATL
5    ATL
6    ATL
7    ATL
8    ATL
9    ATL
Name: TEAM_ABBREVIATION, dtype: str


In [8]:
# print(df["OPP"].dtype)
# print(opp_prev_df["OPP"].dtype)
# print(df["OPP"].head())
# print(opp_prev_df["OPP"].head())

In [9]:
# # Merge the opponent's previous stats back into the main dataframe
# df = pd.merge(df, opp_prev_df, on=["GAME_ID", "OPP"], how="left")

In [10]:
# Get team win streak

# Ensure data is sorted chronologically by team and date
df = df.sort_values(["TEAM_ID", "GAME_DATE"]).reset_index(drop=True)

# 1. Shift the Win column to evaluate past performance (prevents data leakage)
# df["PREV_WIN"] = df.groupby("TEAM_ID")["WIN"].shift(1).fillna(0)

# 2. Create a group ID that changes whenever a team loses (prev_win == 0)
df["streak_id"] = (df["PREV_WIN"] == 0).groupby(df["TEAM_ID"]).cumsum()

# 3. Calculate the running count of consecutive wins within each streak group
df["WIN_STREAK"] = df.groupby(["TEAM_ID", "streak_id"])["PREV_WIN"].cumsum()

# 4. If the team lost their last game, explicitly force the streak to 0
df.loc[df["PREV_WIN"] == 0, "WIN_STREAK"] = 0

# Drop temporary helper columns
df = df.drop(columns=["streak_id"])

In [11]:
# Get team lose streak

# Ensure data is sorted chronologically by team and date
df = df.sort_values(["TEAM_ID", "GAME_DATE"]).reset_index(drop=True)

df["Loss"] = df["WL"].map({"L": 1, "W": 0, "D": 0})

# 2. Shift the Loss column to evaluate past performance (prevents data leakage)
df["prev_loss"] = df.groupby("TEAM_ID")["Loss"].shift(1).fillna(0)

# 3. Create a group ID that changes whenever a team wins or draws (prev_loss == 0)
df["losing_streak_id"] = (df["prev_loss"] == 0).groupby(df["TEAM_ID"]).cumsum()

# 4. Calculate the running count of consecutive losses within each streak group
df["LOSE_STREAK"] = df.groupby(["TEAM_ID", "losing_streak_id"])["prev_loss"].cumsum()

# 5. If the team won or drew their last game, explicitly force the losing streak to 0
df.loc[df["prev_loss"] == 0, "LOSE_STREAK"] = 0

# Drop temporary helper columns
df = df.drop(columns=["Loss", "prev_loss", "losing_streak_id"])

In [12]:
# Check if win streak and lose streak has both values above zero
# Check if any row has both winning and losing streaks active simultaneously (> 0)
invalid_streaks = df[(df["WIN_STREAK"] > 0) & (df["LOSE_STREAK"] > 0)]

if len(invalid_streaks) > 0:
  print(
      f"Logic Error: Found {len(invalid_streaks)} rows where both streaks are"
      " active at the same time."
  )
  print(
      invalid_streaks[
          ["GAME_DATE", "TEAM_NAME", "WL", "WIN_STREAK", "LOSE_STREAK"]
      ].head()
  )
else:
  print("Validation Passed: No rows have overlapping winning and losing streaks.")

# Check for unexpected negative values or nulls
print("Min winning streak:", df["WIN_STREAK"].min())
print("Min losing streak:", df["LOSE_STREAK"].min())
print("Missing values in winning_streak:", df["WIN_STREAK"].isna().sum())
print("Missing values in losing_streak:", df["LOSE_STREAK"].isna().sum())

Validation Passed: No rows have overlapping winning and losing streaks.
Min winning streak: 0.0
Min losing streak: 0.0
Missing values in winning_streak: 30
Missing values in losing_streak: 0


In [13]:
# opp_streak_df = df[
#     ["GAME_ID", "TEAM_ABBREVIATION", "WIN_STREAK", "LOSE_STREAK"]
# ].copy()

# # Rename columns to represent the opposition
# opp_streak_df = opp_streak_df.rename(
#     columns={
#         "TEAM_ABBREVIATION": "OPP",
#         "WIN_STREAK": "OPP_WIN_STREAK",
#         "LOSE_STREAK": "OPP_LOSE_STREAK",
#     }
# )

# # Merge the opponent's streaks back into the main dataframe
# df = pd.merge(df, opp_streak_df, on=["GAME_ID", "OPP"], how="left")

In [14]:
df = df.sort_values(["TEAM_ID", "GAME_DATE"]).reset_index(drop=True)
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])  # if not already datetime

# Days since this team's previous game
df["DAYS_REST"] = df.groupby("TEAM_ID")["GAME_DATE"].diff().dt.days

# Flag back-to-backs specifically (0 or 1 day rest is the meaningful cutoff)
df["IS_BACK_TO_BACK"] = (df["DAYS_REST"] <= 1).astype(int)

In [15]:
# opp_rest_df = df[["GAME_ID", "TEAM_ABBREVIATION", "DAYS_REST"]].copy()
# opp_rest_df = opp_rest_df.rename(
#     columns={"TEAM_ABBREVIATION": "OPP", "DAYS_REST": "OPP_DAYS_REST"}
# )
# df = pd.merge(df, opp_rest_df, on=["GAME_ID", "OPP"], how="left")

In [16]:
# Sort date
df = df.sort_values(by='GAME_DATE', ascending=True)

In [17]:
print(df.columns)

Index(['GAME_DATE', 'GAME_ID', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'TEAM_NAME', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE',
       'OPP', 'WIN', 'HOME', 'PREV_WIN', 'PREV_PTS', 'PREV_PLUSMINUS',
       'WIN_STREAK', 'LOSE_STREAK', 'DAYS_REST', 'IS_BACK_TO_BACK'],
      dtype='str')


In [18]:
# Save to csv
df.to_csv("../preparation/shiftvalues.csv", index=False)
print(df.shape)

(16578, 39)


In [19]:
home_win_rate = df[df['MATCHUP'].str.contains('vs.')]['WIN'].mean()
print(home_win_rate)

0.5517574586302694
